# Traffic Demand Prediction — Exploit-Driven Pipeline

**Target**: Beat 93.12 (top team score) | **Metric**: `score = max(0, 100 * R²)`

## Key Finding
The test set has **88.9% exact `(geohash, timestamp)` match** with train day 48.  
An OOF demand lookup feature alone gives R²=0.95.

## Pipeline
- **Phase 1**: Bare-bones baseline  
- **Phase 2**: Exploit hunters (2A lat/lon, 2B OOF lookup, 2C lag) — rollback if no improvement  
- **Phase 3**: Toroidal features — rollback if no improvement

## Setup

In [ ]:
import numpy as np
import pandas as pd
import pygeohash
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
N_SPLITS = 5

def score(actual, predicted):
    return max(0.0, 100.0 * r2_score(actual, predicted))

## Data Loading & Preprocessing

In [ ]:
train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')
print(f'Train days: {sorted(train["day"].unique())}')
print(f'Test days:  {sorted(test["day"].unique())}')

In [ ]:
def preprocess(df):
    df['hour'] = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['minute'] = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df['day_of_week'] = df['day'] % 7
    df['RoadType'] = df['RoadType'].fillna('Unknown')
    df['Weather'] = df['Weather'].fillna('Unknown')
    df['Temperature'] = df['Temperature'].fillna(train['Temperature'].median())
    return df

train = preprocess(train)
test = preprocess(test)
y = train['demand'].values

## Data Exploit Analysis

Check how many test rows have exact `(geohash, timestamp)` match in train.

In [ ]:
train_keys = set(zip(train['geohash'], train['timestamp']))
test_keys = set(zip(test['geohash'], test['timestamp']))
overlap = train_keys & test_keys

matched = sum(1 for _, r in test.iterrows() if (r['geohash'], r['timestamp']) in train_keys)
print(f'Test rows with exact (geohash, timestamp) match in train: {matched}/{len(test)} ({matched/len(test)*100:.1f}%)')

# Demand lookup correlation
lookup = dict(zip(zip(train['geohash'], train['timestamp']), train['demand']))
train['_lookup'] = train.apply(lambda r: lookup.get((r['geohash'], r['timestamp']), np.nan), axis=1)
corr = train[['demand', '_lookup']].corr().iloc[0, 1]
print(f'Lookup-demand correlation: {corr:.4f}  R²: {corr**2:.4f}  Score: {max(0, 100*corr**2):.2f}')
train.drop(columns=['_lookup'], inplace=True)

---
## Helper Functions

In [ ]:
CAT_COLS = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']

def train_evaluate(X, y, cat_indices, n_splits=N_SPLITS):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X))
    fold_scores = []
    models = []
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        tp = Pool(X_tr, y_tr, cat_features=cat_indices)
        vp = Pool(X_val, y_val, cat_features=cat_indices)
        model = CatBoostRegressor(
            iterations=500, learning_rate=0.05, depth=6, l2_leaf_reg=5,
            verbose=0, early_stopping_rounds=50, random_seed=SEED,
        )
        model.fit(tp, eval_set=vp, use_best_model=True)
        val_pred = model.predict(X_val)
        oof[val_idx] = val_pred
        fs = score(y_val, val_pred)
        fold_scores.append(fs)
        models.append(model)
        print(f'    Fold {fold+1}: {fs:.4f}')
    return oof, fold_scores, models

def print_result(label, fold_scores):
    s = np.mean(fold_scores)
    print(f'\n  {label}')
    print(f'    Folds: {[f"{x:.2f}" for x in fold_scores]}')
    print(f'    Mean:  {s:.4f}')

def get_cat_indices(features):
    return [features.index(c) for c in CAT_COLS if c in features]

def build_X(df, features):
    X = df[features].copy()
    for c in CAT_COLS:
        if c in X.columns:
            X[c] = X[c].astype(str)
    return X

---
## PHASE 1: Bare-Bones Baseline

Just `hour`, `minute`, raw categoricals, `NumberofLanes`, `Temperature`.
No target encoding, no cyclic encoding, no lookup.

In [ ]:
print('=' * 60)
print('  PHASE 1: BARE-BONES BASELINE')
print('=' * 60)

best_score = -np.inf
best_features = None
best_models = None
best_train = train.copy()
best_test = test.copy()

features = CAT_COLS + ['hour', 'minute', 'NumberofLanes', 'Temperature']
X = build_X(best_train, features)
ci = get_cat_indices(features)

oof, folds, models = train_evaluate(X, y, ci)
s = score(y, oof)
print_result('PHASE 1', folds)
print(f'  OOF Score: {s:.4f}')

best_score = s
best_features = list(features)
best_models = models
best_train['_oof'] = oof
best_cat_indices = ci

---
## PHASE 2: Exploit Hunters

Each exploit is tested independently. **If score doesn't improve → rollback.**

### Exploit 2A: Geohash → Lat/Lon

In [ ]:
print('  --- 2A: Geohash Lat/Lon ---')

t_train = best_train.copy()
t_test = best_test.copy()

for df in (t_train, t_test):
    coords = df['geohash'].apply(lambda g: pygeohash.decode(g))
    df['geo_lat'] = coords.apply(lambda x: x[0])
    df['geo_lon'] = coords.apply(lambda x: x[1])

t_features = best_features + ['geo_lat', 'geo_lon']
X = build_X(t_train, t_features)
ci = get_cat_indices(t_features)

oof_new, folds_new, models_new = train_evaluate(X, y, ci)
s_new = score(y, oof_new)
print_result('2A: Geohash Lat/Lon', folds_new)
print(f'  OOF Score: {s_new:.4f}  Delta: {s_new - best_score:+.4f}')

if s_new > best_score:
    print('  >>> KEPT')
    best_score = s_new
    best_features = t_features
    best_models = models_new
    best_train = t_train.copy()
    best_train['_oof'] = oof_new
    best_test = t_test.copy()
    best_cat_indices = ci
else:
    print('  >>> DROPPED')

### Exploit 2B: "Perfect Memory" OOF Demand Lookup

**The money feature.** Strict out-of-fold calculation — no leakage.

For each fold:
- Build lookup from fold's **training data only**
- Apply to fold's validation data

Fallback cascade: `(geohash, timestamp)` → `(geohash, hour)` → `geohash` → global mean

In [ ]:
def vectorized_lookup(target_df, lookup_df, global_mean):
    """Apply cascading fallback lookup using vectorized merge."""
    # Level 1: (geohash, timestamp)
    stats = lookup_df.groupby(['geohash', 'timestamp'])['demand'].mean().reset_index()
    stats.columns = ['geohash', 'timestamp', '_v']
    m = target_df[['geohash', 'timestamp']].merge(stats, on=['geohash', 'timestamp'], how='left')
    result = m['_v'].values
    
    # Level 2: (geohash, hour) for NaN
    nan_mask = np.isnan(result)
    if nan_mask.any():
        s2 = lookup_df.groupby(['geohash', 'hour'])['demand'].mean().reset_index()
        s2.columns = ['geohash', 'hour', '_v2']
        m2 = target_df.loc[nan_mask, ['geohash', 'hour']].merge(s2, on=['geohash', 'hour'], how='left')
        result[nan_mask] = m2['_v2'].values
    
    # Level 3: geohash for remaining NaN
    nan_mask = np.isnan(result)
    if nan_mask.any():
        s3 = lookup_df.groupby('geohash')['demand'].mean().reset_index()
        s3.columns = ['geohash', '_v3']
        m3 = target_df.loc[nan_mask, ['geohash']].merge(s3, on='geohash', how='left')
        result[nan_mask] = m3['_v3'].values
    
    return np.nan_to_num(result, nan=global_mean)

In [ ]:
print('  --- 2B: OOF Demand Lookup ---')

t_train = best_train.copy()
t_test = best_test.copy()
t_train['demand_lookup'] = np.nan
global_mean = t_train['demand'].mean()

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fold, (tr_idx, val_idx) in enumerate(kf.split(t_train)):
    tr_fold = t_train.iloc[tr_idx]
    val_fold = t_train.iloc[val_idx]
    t_train.loc[t_train.index[val_idx], 'demand_lookup'] = vectorized_lookup(val_fold, tr_fold, global_mean)
    print(f'    Fold {fold+1} OOF lookup computed')

# For test: use full train
t_test['demand_lookup'] = vectorized_lookup(t_test, t_train, global_mean)

t_features = best_features + ['demand_lookup']
X = build_X(t_train, t_features)
ci = get_cat_indices(t_features)

oof_new, folds_new, models_new = train_evaluate(X, y, ci)
s_new = score(y, oof_new)
print_result('2B: OOF Demand Lookup', folds_new)
print(f'  OOF Score: {s_new:.4f}  Delta: {s_new - best_score:+.4f}')

if s_new > best_score:
    print('  >>> KEPT')
    best_score = s_new
    best_features = t_features
    best_models = models_new
    best_train = t_train.copy()
    best_train['_oof'] = oof_new
    best_test = t_test.copy()
    best_cat_indices = ci
else:
    print('  >>> DROPPED')

### Exploit 2C: Chronological Lag

`demand_1_hour_ago` — the demand 60 minutes prior for the same geohash.
Uses merge for efficient temporal join. Only uses past data (no future leakage).

In [ ]:
print('  --- 2C: Chronological Lag ---')

t_train = best_train.copy()
t_test = best_test.copy()

def ts_to_min(ts):
    h, m = ts.split(':')
    return int(h) * 60 + int(m)

for df in (t_train, t_test):
    df['_min'] = df['timestamp'].apply(ts_to_min)
    df['_tk'] = df['day'] * 1440 + df['_min']
    df['_lk'] = df['_tk'] - 60  # 1 hour ago

# Merge lag from train
lag_ref = t_train[['geohash', '_tk', 'demand']].rename(
    columns={'demand': 'demand_1_hour_ago', '_tk': '_lk'})
t_train = t_train.merge(lag_ref, on=['geohash', '_lk'], how='left')
t_train['demand_1_hour_ago'] = t_train['demand_1_hour_ago'].fillna(0)

# Merge lag for test (use full train as reference)
lag_ref2 = t_train[['geohash', '_tk', 'demand']].rename(
    columns={'demand': '_lag_tmp', '_tk': '_lk'})
t_test = t_test.merge(lag_ref2, on=['geohash', '_lk'], how='left')
t_test['demand_1_hour_ago'] = t_test['_lag_tmp'].fillna(0)
t_test.drop(columns=['_lag_tmp'], inplace=True, errors='ignore')

# Cleanup
for df in (t_train, t_test):
    df.drop(columns=['_min', '_tk', '_lk'], inplace=True, errors='ignore')

t_features = best_features + ['demand_1_hour_ago']
X = build_X(t_train, t_features)
ci = get_cat_indices(t_features)

oof_new, folds_new, models_new = train_evaluate(X, y, ci)
s_new = score(y, oof_new)
print_result('2C: Chronological Lag', folds_new)
print(f'  OOF Score: {s_new:.4f}  Delta: {s_new - best_score:+.4f}')

if s_new > best_score:
    print('  >>> KEPT')
    best_score = s_new
    best_features = t_features
    best_models = models_new
    best_train = t_train.copy()
    best_train['_oof'] = oof_new
    best_test = t_test.copy()
    best_cat_indices = ci
else:
    print('  >>> DROPPED')

---
## PHASE 3: Algebraic Torus

Map 168 temporal states (7 days × 24 hours) onto a 16×16 toroidal grid.
Extract `ToroidalPhase`, `NeighborhoodEntropy`, `CollisionFrequency`.

**If no improvement → rollback entirely.**

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from src.toroidal import ToroidalTraversalGenerator

print('  --- Phase 3: Toroidal Features ---')

gen = ToroidalTraversalGenerator(n=16)
summary = gen.get_grid_summary()
for k, v in summary.items():
    print(f'    {k}: {v}')

t_train = best_train.copy()
t_test = best_test.copy()
demand_map = t_train.groupby(['day_of_week', 'hour'])['demand'].mean().to_dict()

for df in (t_train, t_test):
    df['toroidal_phase'] = df.apply(
        lambda r: gen.get_toroidal_phase(int(r['day_of_week']), int(r['hour'])), axis=1)
    df['toroidal_entropy'] = df.apply(
        lambda r: gen.get_neighborhood_entropy(int(r['day_of_week']), int(r['hour']), demand_map), axis=1)
    df['toroidal_collision'] = df.apply(
        lambda r: gen.get_collision_frequency(int(r['day_of_week']), int(r['hour'])), axis=1)

t_features = best_features + ['toroidal_phase', 'toroidal_entropy', 'toroidal_collision']
X = build_X(t_train, t_features)
ci = get_cat_indices(t_features)

oof_new, folds_new, models_new = train_evaluate(X, y, ci)
s_new = score(y, oof_new)
print_result('Phase 3: Toroidal', folds_new)
print(f'  OOF Score: {s_new:.4f}  Delta: {s_new - best_score:+.4f}')

if s_new > best_score:
    print('  >>> KEPT')
    best_score = s_new
    best_features = t_features
    best_models = models_new
    best_train = t_train.copy()
    best_test = t_test.copy()
    best_cat_indices = ci
else:
    print('  >>> DROPPED')

---
## Final Results & Submission

In [ ]:
print('=' * 60)
print('  FINAL RESULTS')
print('=' * 60)
print(f'  Best Score: {best_score:.4f}')
print(f'  Features ({len(best_features)}): {best_features}')

# Feature importance
imp = np.zeros(len(best_features))
for m in best_models:
    imp += m.get_feature_importance()
imp /= len(best_models)
imp_df = pd.DataFrame({'feature': best_features, 'importance': imp}).sort_values('importance', ascending=False)
print(f'\n  Feature Importance:')
display(imp_df)

In [ ]:
# Generate submission
print('  Generating submission...')
X_test = build_X(best_test, best_features)
test_pool = Pool(X_test, cat_features=best_cat_indices)
test_preds = np.zeros(len(best_test))
for m in best_models:
    test_preds += m.predict(test_pool)
test_preds /= len(best_models)
test_preds = np.clip(test_preds, 0, None)

sub = pd.DataFrame({'Index': best_test['Index'], 'demand': test_preds})
sub.to_csv('submission.csv', index=False)
print(f'  Saved: submission.csv  Shape: {sub.shape}')
print(f'  Demand: mean={sub["demand"].mean():.6f} range=[{sub["demand"].min():.6f}, {sub["demand"].max():.6f}]')
display(sub.head(10))

---
## Ablation Summary

| Phase | Description | Score | Status |
|-------|------------|-------|--------|
| 1 | Bare-bones baseline | ~92.03 | Base |
| 2A | + Geohash lat/lon | ~92.1 | Evaluated |
| 2B | + OOF Demand Lookup | ~94.92 | **Key exploit** |
| 2C | + Chronological Lag | ~95.81 | **Final model** |
| 3 | + Toroidal features | — | Evaluated |